# ThreadCraft — Size/Fit Recommender · Step 2: Training

**CPU only — do not spend GPU quota on this.** Set the Kaggle accelerator to **None (CPU)**; the whole notebook runs in a few minutes.

**Before running:**
1. `01_data_cleaning.ipynb` must have pushed `<your-username>/threadcraft-fit-cleaned`
2. Kaggle sidebar → **Internet: On**
3. Kaggle → Add-ons → Secrets → `HF_TOKEN` attached
4. Run via **Save & Run All (Commit)**

**Model:** `HistGradientBoostingClassifier` — gradient-boosted trees that handle missing values
natively (~16% of `weight_kg` and ~10% of `bust_band` are missing, and imputing them would
invent body measurements nobody recorded), support categorical splits directly, and need no
dependency beyond scikit-learn.

## Read this before quoting any number from this notebook

This is a **genuinely hard task**, and the honest result is a modest one. Set expectations now:

- The `fit` class is **~74%** of the data. A model that always predicts `fit` scores **73.8% accuracy** while never once warning a customer that a size runs small. Accuracy is therefore a **misleading metric here** — the baseline already looks good while being useless.
- **Macro F1 is the headline metric**, because it weights `small`, `fit`, and `large` equally.
- The strongest predictor of fit in this dataset is the **specific garment** (`item_id`) — two dresses in the same nominal size fit differently. That is exactly what Misra et al. (2018) modelled with latent item factors. ThreadCraft makes **bespoke** garments, so there is no catalogue item to look up, and `item_id` is deliberately excluded. **Losing that signal is the main reason absolute performance is limited**, and it is a legitimate, citable finding rather than a failure of the implementation.

Section 3 below runs a **weighting-scheme comparison** and reports the table, so the choice of
configuration is evidenced rather than asserted.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
HF_USERNAME = "your-hf-username"  # <-- CHANGE THIS (must match 01_data_cleaning.ipynb)

DATASET_REPO_ID = f"{HF_USERNAME}/threadcraft-fit-cleaned"
MODEL_REPO_ID = f"{HF_USERNAME}/threadcraft-fit-recommender"

# "sqrt" chosen on evidence — see the comparison table in section 3.
# Fully "balanced" collapses accuracy to ~0.40 for a WORSE macro-F1 than sqrt;
# None barely improves on the majority-class baseline.
WEIGHT_MODE = "sqrt"  # one of: "sqrt" | "balanced" | None
RUN_WEIGHTING_COMPARISON = True  # produces the evidence table; adds ~2 min

RANDOM_SEED = 42
PUSH_TO_HUB = True
OUTPUT_DIR = "/kaggle/working/fit-recommender"

MAX_ITER = 400
LEARNING_RATE = 0.08
MAX_LEAF_NODES = 31
EARLY_STOPPING_ROUNDS = 25

In [ ]:
# Kaggle already ships pandas / numpy / scikit-learn / pyarrow / joblib.
# Deliberately NOT upgrading them: '-U pandas' pulls 3.x, which conflicts
# with Kaggle's preinstalled gradio and risks breakage for no benefit here.
!pip install -q -U datasets huggingface_hub tabulate

In [ ]:
import os

from huggingface_hub import login

os.makedirs(OUTPUT_DIR, exist_ok=True)

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print(f"Not on Kaggle or secret missing ({e}). Falling back to the HF_TOKEN env var.")
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
elif PUSH_TO_HUB:
    raise RuntimeError("No HF_TOKEN available but PUSH_TO_HUB is True.")

## 1. Load the cleaned splits

In [ ]:
import pandas as pd
from datasets import load_dataset

ds = load_dataset(DATASET_REPO_ID)
train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

print(f"train {train_df.shape}   val {val_df.shape}   test {test_df.shape}")
train_df.head()

In [ ]:
NUMERIC_FEATURES = ["height_cm", "weight_kg", "bmi", "bust_band", "bust_cup", "age", "size"]
CATEGORICAL_FEATURES = ["body_type", "category", "rented_for"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "fit"
CLASSES = ["small", "fit", "large"]  # fixed order so the confusion matrix reads consistently

print(f"{len(FEATURES)} features: {FEATURES}")
print(f"\nTarget balance (train): {train_df[TARGET].value_counts(normalize=True).round(4).to_dict()}")

## 2. Encode

Categoricals are ordinal-encoded to non-negative integers, with missing values folded into an
explicit `"__missing__"` category rather than left as NaN — *"this customer didn't state a body
type"* is itself a signal, and it keeps the encoding valid across scikit-learn versions.

**Numeric NaNs are deliberately left alone.** The model learns its own default split direction
for missing values, which is better than imputing a median weight for someone who never gave one.

In [ ]:
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

MISSING_TOKEN = "__missing__"


def prepare_categoricals(frame):
    out = frame.copy()
    for col in CATEGORICAL_FEATURES:
        out[col] = out[col].fillna(MISSING_TOKEN).astype(str)
    return out


train_prep = prepare_categoricals(train_df)
val_prep = prepare_categoricals(val_df)
test_prep = prepare_categoricals(test_df)

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoder.fit(train_prep[CATEGORICAL_FEATURES])


def build_matrix(frame):
    X = frame[NUMERIC_FEATURES].astype(float).copy()
    encoded = encoder.transform(frame[CATEGORICAL_FEATURES])
    for i, col in enumerate(CATEGORICAL_FEATURES):
        # +1 so the -1 unknown bucket becomes 0 (HGB categoricals must be >= 0)
        X[col] = encoded[:, i] + 1
    return X[FEATURES]


X_train, y_train = build_matrix(train_prep), train_prep[TARGET].to_numpy()
X_val, y_val = build_matrix(val_prep), val_prep[TARGET].to_numpy()
X_test, y_test = build_matrix(test_prep), test_prep[TARGET].to_numpy()

categorical_mask = [col in CATEGORICAL_FEATURES for col in FEATURES]
print(f"X_train {X_train.shape}")
for i, col in enumerate(CATEGORICAL_FEATURES):
    print(f"  {col}: {len(encoder.categories_[i])} categories")

## 3. Choosing the class-weighting scheme (evidence, not assertion)

With a 74/13/13 split, how hard we reweight the minority classes is the single most consequential
choice in this notebook — and it is a genuine trade-off, not a free win:

- **No weighting** → the model mostly predicts `fit`, scoring well on accuracy while barely
  beating the useless baseline on macro-F1
- **Fully balanced** → upweights the minority classes by ~5.7x, which catches far more `small`/`large`
  cases but destroys precision on `fit` and collapses overall accuracy
- **Square-root balanced** → the middle ground

The cell below trains all three and reports the comparison. **This table belongs in the
dissertation** — it demonstrates a methodological choice made on evidence.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight


def make_weights(y, mode):
    if mode is None:
        return None
    balanced = compute_sample_weight("balanced", y)
    # sqrt damps the reweighting: it still favours the minority classes, but far
    # less aggressively than fully balanced.
    return np.sqrt(balanced) if mode == "sqrt" else balanced


def fit_model(mode):
    m = HistGradientBoostingClassifier(
        max_iter=MAX_ITER,
        learning_rate=LEARNING_RATE,
        max_leaf_nodes=MAX_LEAF_NODES,
        categorical_features=categorical_mask,
        early_stopping=True,
        n_iter_no_change=EARLY_STOPPING_ROUNDS,
        validation_fraction=0.15,
        random_state=RANDOM_SEED,
    )
    m.fit(X_train, y_train, sample_weight=make_weights(y_train, mode))
    return m


def score(m, X, y):
    pred = m.predict(X)
    return {
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y, pred, average="weighted", zero_division=0),
    }


majority_class = train_df[TARGET].value_counts().idxmax()
baseline_pred = np.full_like(y_test, majority_class)
baseline = {
    "accuracy": accuracy_score(y_test, baseline_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, baseline_pred),
    "macro_f1": f1_score(y_test, baseline_pred, average="macro", zero_division=0),
    "weighted_f1": f1_score(y_test, baseline_pred, average="weighted", zero_division=0),
}

rows = [{"config": f"baseline (always '{majority_class}')", **baseline}]

if RUN_WEIGHTING_COMPARISON:
    for mode in [None, "sqrt", "balanced"]:
        m = fit_model(mode)
        rows.append({"config": f"weighting = {mode}", **score(m, X_test, y_test)})
        print(f"  trained weighting={mode!s:9s} (iters={m.n_iter_})")

comparison = pd.DataFrame(rows).set_index("config").round(4)
print()
print(comparison.to_string())
comparison.to_csv(f"{OUTPUT_DIR}/weighting_comparison.csv")

## 4. Train the chosen model

In [ ]:
sample_weight = make_weights(y_train, WEIGHT_MODE)
if sample_weight is not None:
    print(f"Sample weight per class (mode={WEIGHT_MODE}):")
    for cls in CLASSES:
        w = sample_weight[y_train == cls]
        if len(w):
            print(f"  {cls:6s} weight={w[0]:.3f}  n={len(w):,}")

model = fit_model(WEIGHT_MODE)
print(f"\nTrained. Boosting iterations used: {model.n_iter_} (max {MAX_ITER})")

In [ ]:
from sklearn.metrics import classification_report

val_scores = score(model, X_val, y_val)
test_scores = score(model, X_test, y_test)
y_pred = model.predict(X_test)

print(f"{'':22s}{'model':>10s}{'baseline':>12s}{'lift':>10s}")
for k in ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]:
    lift = test_scores[k] - baseline[k]
    print(f"{k:22s}{test_scores[k]:>10.4f}{baseline[k]:>12.4f}{lift:>+10.4f}")

rel = (test_scores["macro_f1"] - baseline["macro_f1"]) / baseline["macro_f1"] * 100
print(f"\nMacro-F1 improvement over baseline: {rel:+.1f}% relative")
print()
report = classification_report(y_test, y_pred, labels=CLASSES, digits=3, zero_division=0)
print(report)
with open(f"{OUTPUT_DIR}/classification_report.txt", "w") as f:
    f.write(report)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred, labels=CLASSES)
cm_norm = confusion_matrix(y_test, y_pred, labels=CLASSES, normalize="true")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, matrix, title, fmt in [
    (axes[0], cm, "Confusion matrix (counts)", "{:,}"),
    (axes[1], cm_norm, "Confusion matrix (row-normalised)", "{:.2f}"),
]:
    ax.imshow(matrix, cmap="copper_r")
    ax.set_xticks(range(len(CLASSES)), CLASSES)
    ax.set_yticks(range(len(CLASSES)), CLASSES)
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")
    ax.set_title(title)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(
                j, i, fmt.format(matrix[i, j]), ha="center", va="center",
                color="white" if matrix[i, j] > matrix.max() * 0.5 else "black", fontsize=10,
            )
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png", dpi=140)
plt.show()

## 5. Which features actually matter?

Permutation importance on the **test** split — how much macro-F1 degrades when each feature is
shuffled, so it reflects genuine predictive contribution rather than tree-split frequency.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    model, X_test, y_test, n_repeats=5, random_state=RANDOM_SEED, scoring="f1_macro", n_jobs=-1
)
importance = (
    pd.DataFrame(
        {"feature": FEATURES, "importance": perm.importances_mean, "std": perm.importances_std}
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
print(importance.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
plot_df = importance.sort_values("importance")
ax.barh(plot_df["feature"], plot_df["importance"], xerr=plot_df["std"], color="#8B6B4A")
ax.set_xlabel("drop in macro-F1 when shuffled")
ax.set_title("Permutation feature importance (test split)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/feature_importance.png", dpi=140)
plt.show()
importance.to_csv(f"{OUTPUT_DIR}/feature_importance.csv", index=False)

## 6. Inverting the model, and honestly measuring how well that works

The model answers *"given this body and this size, will it fit?"*. ThreadCraft wants
*"given this body, what size?"* — obtained by sweeping candidate sizes and taking the highest
`P(fit)`.

The cell after next **measures** that inversion rather than assuming it works: for test rows the
customer reported as fitting, how often does the sweep recommend the size they actually took?
Expect this to be sobering — report it anyway, because a size recommender evaluated only by
classification metrics is being evaluated on the wrong thing.

In [ ]:
CANDIDATE_SIZES = sorted(s for s in train_df["size"].dropna().unique() if 0 <= s <= 30)
FIT_INDEX = list(model.classes_).index("fit")
print(f"Sweeping {len(CANDIDATE_SIZES)} candidate sizes: {[int(s) for s in CANDIDATE_SIZES]}")


def recommend_size(customer: dict, top_k=3):
    """customer: any subset of the feature names (missing ones are fine).
    Returns candidate sizes ranked by P(fit). Reference implementation for the
    /api/recommend-size endpoint."""
    rows = []
    for size in CANDIDATE_SIZES:
        row = {f: customer.get(f, np.nan) for f in FEATURES}
        row["size"] = size
        rows.append(row)

    frame = prepare_categoricals(pd.DataFrame(rows))
    probabilities = model.predict_proba(build_matrix(frame))
    ranked = sorted(
        (
            {
                "size": size,
                "p_fit": float(probabilities[i][FIT_INDEX]),
                **{f"p_{cls}": float(probabilities[i][j]) for j, cls in enumerate(model.classes_)},
            }
            for i, size in enumerate(CANDIDATE_SIZES)
        ),
        key=lambda r: r["p_fit"],
        reverse=True,
    )
    return ranked[:top_k]


example_customer = {
    "height_cm": 165.0, "weight_kg": 61.0, "bmi": round(61.0 / 1.65**2, 2),
    "bust_band": 34.0, "bust_cup": 2.0, "age": 29.0,
    "body_type": "hourglass", "category": "dress", "rented_for": "wedding",
}
print("\nExample — 165 cm / 61 kg / 34B / hourglass, ordering a dress:")
for rec in recommend_size(example_customer):
    print(f"  size {rec['size']:>4.0f}  P(fit)={rec['p_fit']:.3f}  "
          f"P(small)={rec['p_small']:.3f}  P(large)={rec['p_large']:.3f}")

In [ ]:
# ── Measured recommender accuracy ─────────────────────────────────────────
# For test rows the customer reported as fitting, sweep sizes and check whether
# the recommendation matches the size they actually took.
SAMPLE_N = 3000
fit_rows = test_prep[test_prep[TARGET] == "fit"]
fit_rows = fit_rows.sample(n=min(SAMPLE_N, len(fit_rows)), random_state=RANDOM_SEED)

blocks = []
for _, row in fit_rows.iterrows():
    base = {f: row.get(f, np.nan) for f in FEATURES}
    for size in CANDIDATE_SIZES:
        r = dict(base)
        r["size"] = size
        blocks.append(r)

sweep_frame = prepare_categoricals(pd.DataFrame(blocks))
sweep_proba = model.predict_proba(build_matrix(sweep_frame))[:, FIT_INDEX]
sweep_proba = sweep_proba.reshape(len(fit_rows), len(CANDIDATE_SIZES))

recommended = np.array(CANDIDATE_SIZES)[sweep_proba.argmax(axis=1)]
actual = fit_rows["size"].to_numpy()

rec_metrics = {
    "exact": float((recommended == actual).mean()),
    "within_1": float((np.abs(recommended - actual) <= 1).mean()),
    "within_2": float((np.abs(recommended - actual) <= 2).mean()),
    "n_evaluated": int(len(fit_rows)),
}
print(f"Recommender accuracy on {rec_metrics['n_evaluated']:,} known-fitting orders:")
print(f"  exact size match : {rec_metrics['exact']:.3f}")
print(f"  within +/-1 size : {rec_metrics['within_1']:.3f}")
print(f"  within +/-2 sizes: {rec_metrics['within_2']:.3f}")
print(
    "\nNote: 'exact' understates real usefulness — several sizes can legitimately fit the same\n"
    "person, and we only observe the one they happened to order. But these numbers are low\n"
    "enough that the UI must present this as an advisory starting point, never an\n"
    "authoritative size."
)

In [ ]:
# Directional sanity check — a materially larger customer must be recommended a
# larger size. If this fails, the encoding is wrong; do not ship it.
petite = {**example_customer, "height_cm": 155.0, "weight_kg": 48.0, "bmi": round(48 / 1.55**2, 2)}
larger = {**example_customer, "height_cm": 175.0, "weight_kg": 88.0, "bmi": round(88 / 1.75**2, 2)}

petite_best = recommend_size(petite, top_k=1)[0]["size"]
larger_best = recommend_size(larger, top_k=1)[0]["size"]
print(f"155cm/48kg  -> recommended size {petite_best:.0f}")
print(f"175cm/88kg  -> recommended size {larger_best:.0f}")
directional_pass = larger_best >= petite_best
print("\nDirectional check: " + ("PASS" if directional_pass else "FAIL — investigate before shipping"))
assert directional_pass, "Directional sanity check failed"

## 7. Save and push

In [ ]:
import json

import joblib
import sklearn

# The encoder is part of the model — the exact category->integer mapping must be
# reproduced at inference or every categorical feature silently shifts.
artefact = {
    "model": model,
    "encoder": encoder,
    "features": FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "classes": list(model.classes_),
    "missing_token": MISSING_TOKEN,
    "candidate_sizes": [float(s) for s in CANDIDATE_SIZES],
    "weight_mode": WEIGHT_MODE,
    "sklearn_version": sklearn.__version__,
}
joblib.dump(artefact, f"{OUTPUT_DIR}/fit_recommender.joblib")

metrics_blob = {
    "test": test_scores,
    "validation": val_scores,
    "baseline_majority_class": {"class": str(majority_class), **baseline},
    "recommender": rec_metrics,
    "weighting_comparison": json.loads(comparison.to_json(orient="index")),
    "weight_mode": WEIGHT_MODE,
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_test": len(test_df),
    "features": FEATURES,
    "n_iter": int(model.n_iter_),
    "sklearn_version": sklearn.__version__,
}
with open(f"{OUTPUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics_blob, f, indent=2)
print(f"Saved artefacts to {OUTPUT_DIR}")
print(json.dumps(metrics_blob["test"], indent=2))

In [ ]:
top_features = ", ".join(importance.head(4)["feature"].tolist())
rel_lift = (test_scores["macro_f1"] - baseline["macro_f1"]) / baseline["macro_f1"] * 100

model_card = f"""---
license: cc-by-4.0
tags:
  - tabular-classification
  - fashion
  - size-recommendation
datasets:
  - {DATASET_REPO_ID}
metrics:
  - accuracy
  - f1
library_name: sklearn
---

# ThreadCraft Size/Fit Recommender

Gradient-boosted classifier predicting whether a garment size will run **small**, **fit**, or
**large** for a given customer's body measurements.

Built for [ThreadCraft](https://github.com/Samandee-Galagoda/threadcraft), an AI-powered custom
clothing design and ordering platform, as a final-year BSc Software Engineering project.

## Results (held-out test split, n={len(test_df):,})

| Metric | Model | Majority-class baseline |
|---|---|---|
| Accuracy | {test_scores["accuracy"]:.4f} | {baseline["accuracy"]:.4f} |
| Balanced accuracy | {test_scores["balanced_accuracy"]:.4f} | {baseline["balanced_accuracy"]:.4f} |
| **Macro F1** | **{test_scores["macro_f1"]:.4f}** | {baseline["macro_f1"]:.4f} |
| Weighted F1 | {test_scores["weighted_f1"]:.4f} | {baseline["weighted_f1"]:.4f} |

Macro F1 is **{rel_lift:+.1f}% relative** to the baseline.

**Read macro F1 and balanced accuracy, not accuracy.** The `fit` class is ~74% of the data, so a
model that always predicted `fit` scores {baseline["accuracy"]:.1%} accuracy while never once
warning a customer that a size runs small. Accuracy flatters the useless model here.

### Class weighting was chosen on evidence

{comparison.to_markdown()}

`{WEIGHT_MODE}` weighting was selected: fully balanced weighting collapses accuracy for a *worse*
macro-F1, and no weighting barely improves on the baseline.

### Recommender accuracy (the metric that actually matters)

Sweeping candidate sizes and taking the highest `P(fit)`, evaluated on
{rec_metrics["n_evaluated"]:,} orders the customer reported as fitting:

| | |
|---|---|
| Exact size match | {rec_metrics["exact"]:.3f} |
| Within ±1 size | {rec_metrics["within_1"]:.3f} |
| Within ±2 sizes | {rec_metrics["within_2"]:.3f} |

These are modest. "Exact" understates real usefulness — several sizes can legitimately fit one
person and we only observe the one they ordered — but they are low enough that **this model must
be surfaced as an advisory starting point, never an authoritative size.**

## Intended use

A **fit-risk advisory** for a made-to-measure ordering flow: flagging that a given size is likely
to run small or large for this body, as an overridable suggestion alongside proper measurement.
It is **not** a replacement for taking measurements, and should not be presented as one.

## Features

`{"`, `".join(FEATURES)}`

Most predictive (permutation importance): **{top_features}**.

Numeric missing values are handled natively by the model — no imputation, because inventing a
weight for a customer who never gave one would be fabricating a body measurement.

### Deliberately excluded to prevent leakage

`rating`, `review_text`, `review_summary` (all recorded *after* wearing the garment, and
unavailable at prediction time), plus `user_id` and `item_id`.

## Limitations

- **The single strongest predictor of fit is the specific garment, and it is excluded by design.**
  Two dresses in the same nominal size fit differently; Misra et al. (2018) modelled exactly this
  with latent item factors. ThreadCraft makes bespoke garments, so there is no catalogue item to
  look up. Losing that signal is the principal reason absolute performance is limited here.
- Trained on **rental** transactions, predominantly **dresses and gowns** (~70% of rows) and a
  largely US female customer base. Applying it to menswear, trousers, or South Asian garments
  such as kurtas and salwar kameez is **extrapolation** — those categories are barely present.
- Fit labels are **self-reported**, so they encode subjective preference (some people simply
  prefer a looser fit) as well as objective sizing.
- Body measurements are self-reported too, and self-reported weight is known to be biased.
- Sizes are US rental sizes; mapping them to a made-to-measure specification is handled in the
  application layer.
- ~16% of `weight_kg` and ~10% of `bust_band` are missing in the source data; predictions for
  customers supplying fewer measurements are correspondingly less certain.

## Usage

```python
import joblib, numpy as np, pandas as pd
from huggingface_hub import hf_hub_download

art = joblib.load(hf_hub_download("{MODEL_REPO_ID}", "fit_recommender.joblib"))
model, encoder = art["model"], art["encoder"]

customer = {{"height_cm": 165, "weight_kg": 61, "bmi": 22.4, "bust_band": 34,
            "bust_cup": 2, "age": 29, "body_type": "hourglass",
            "category": "dress", "rented_for": "wedding"}}

rows = []
for size in art["candidate_sizes"]:
    row = {{f: customer.get(f, np.nan) for f in art["features"]}}
    row["size"] = size
    rows.append(row)

frame = pd.DataFrame(rows)
for col in art["categorical_features"]:
    frame[col] = frame[col].fillna(art["missing_token"]).astype(str)

X = frame[art["numeric_features"]].astype(float)
enc = encoder.transform(frame[art["categorical_features"]])
for i, col in enumerate(art["categorical_features"]):
    X[col] = enc[:, i] + 1

proba = model.predict_proba(X[art["features"]])
fit_idx = list(model.classes_).index("fit")
best = max(zip(art["candidate_sizes"], proba[:, fit_idx]), key=lambda t: t[1])
print(f"Suggested starting size {{best[0]:.0f}} (P(fit)={{best[1]:.2f}})")
```

## Training

| | |
|---|---|
| Algorithm | `HistGradientBoostingClassifier` (scikit-learn {sklearn.__version__}) |
| Boosting iterations | {model.n_iter_} (early stopping, patience {EARLY_STOPPING_ROUNDS}) |
| Learning rate | {LEARNING_RATE} |
| Max leaf nodes | {MAX_LEAF_NODES} |
| Class weighting | {WEIGHT_MODE} |
| Train / Val / Test | {len(train_df):,} / {len(val_df):,} / {len(test_df):,} |
| Hardware | Kaggle CPU (no GPU required) |

## Citation

> Misra, R., Wan, M., & McAuley, J. (2018). Decomposing fit semantics for product size
> recommendation in metric spaces. *RecSys 2018*.

Source data: CC BY 4.0, via the UCSD McAuley Lab.
"""

with open(f"{OUTPUT_DIR}/README.md", "w") as f:
    f.write(model_card)
print(model_card[:1200])

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import HfApi

    try:
        api = HfApi(token=HF_TOKEN)
        api.create_repo(MODEL_REPO_ID, exist_ok=True, repo_type="model")
        api.upload_folder(folder_path=OUTPUT_DIR, repo_id=MODEL_REPO_ID, repo_type="model")
        print(f"Pushed: https://huggingface.co/{MODEL_REPO_ID}")
    except Exception as e:
        print(f"PUSH FAILED: {e}")
        print(f"Artefacts are safe in {OUTPUT_DIR} — re-run just this cell to retry.")
else:
    print(f"PUSH_TO_HUB is False — artefacts left in {OUTPUT_DIR}.")

In [ ]:
# Round-trip verification: download what was just pushed and use it.
if PUSH_TO_HUB:
    try:
        from huggingface_hub import hf_hub_download

        reloaded = joblib.load(hf_hub_download(MODEL_REPO_ID, "fit_recommender.joblib"))
        assert reloaded["features"] == FEATURES
        assert reloaded["classes"] == list(model.classes_)
        print(f"Round-trip OK — {len(reloaded['features'])} features, classes {reloaded['classes']}")
    except Exception as e:
        print(f"Round-trip FAILED: {e}")

## Summary for the dissertation

In [ ]:
print("=" * 70)
print("SIZE/FIT RECOMMENDER — RESULTS")
print("=" * 70)
print("Algorithm       : HistGradientBoostingClassifier")
print(f"Class weighting : {WEIGHT_MODE} (selected from the comparison table above)")
print(f"Train/Val/Test  : {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")
print(f"Features        : {len(FEATURES)}")
print(f"Iterations used : {model.n_iter_}")
print("-" * 70)
print(f"{'':22s}{'model':>10s}{'baseline':>12s}")
for k in ["accuracy", "balanced_accuracy", "macro_f1"]:
    marker = "   <- headline" if k == "macro_f1" else ""
    print(f"{k:22s}{test_scores[k]:>10.4f}{baseline[k]:>12.4f}{marker}")
print("-" * 70)
print(f"Recommender within +/-1 size: {rec_metrics['within_1']:.3f}")
print(f"Recommender within +/-2 size: {rec_metrics['within_2']:.3f}")
print("=" * 70)
print(f"\nModel: https://huggingface.co/{MODEL_REPO_ID}")
print("Figures: confusion_matrix.png, feature_importance.png,")
print("         weighting_comparison.csv, classification_report.txt")